# Vision Inspection Demo — Synthetic Kerf Dataset + NEU Steel

Demonstrates the C++ `detect_kerf_width` / `detect_blade_tip` / `measure_chipping` kernels on:

1. **Synthetic kerf images** generated by `vision/generate_synthetic_kerf.py`
2. **NEU Surface Defect Dataset** (scratches → kerf edge analog) via `vision/neu_steel_adapter.py`

---
**Physical model** for synthetic images:

| Layer | Brightness | Source |
|---|---|---|
| Silicon substrate | ~140 | Diffuse Si reflectance |
| Kerf void | ~18 | No material → dark |
| Kerf wall | ~220 | Specular reflection from vertical wall |
| Chipping | ~5–30 | Missing Si → shadow/void |
| Blade tip | ~230 | Bright ellipse (leading edge contact) |

Real DISCO DFL7160: blade width 15–200 μm, camera ~5–20 μm/px. Demo uses 10 px/mm for visibility.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Robust REPO_ROOT detection
def _find_repo_root(start=None):
    p = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.isfile(os.path.join(p, 'pyproject.toml')):
            return p
        p = os.path.dirname(p)
    raise RuntimeError('Could not find repo root')

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from vision.generate_synthetic_kerf import KerfImageConfig, generate_kerf_image, make_dataset
from vision.neu_steel_adapter import NeuSteelAdapter, CATEGORY_MAPPING
from vision import detect_kerf_width, detect_blade_tip, measure_chipping, _HAS_CPP

print(f'C++ kernel available: {_HAS_CPP}')
print(f'REPO_ROOT: {REPO_ROOT}')

## 1 — Single synthetic kerf image

In [ ]:
cfg = KerfImageConfig(
    H=128, W=256,
    kerf_width_mm=2.5, pixel_per_mm=10.0,
    chipping_severity=0.3,
    include_blade_tip=True,
    blade_wear=0.2,
    seed=42,
)
img, gt = generate_kerf_image(cfg)

# Run C++ detectors
res_w = detect_kerf_width(img, threshold=5000.0, pixel_per_mm=10.0)
res_t = detect_blade_tip(img, threshold=200.0)
res_c = measure_chipping(img,
                          kerf_left_px=gt['kerf_left_px'],
                          kerf_right_px=gt['kerf_right_px'],
                          pixel_per_mm=10.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: image with overlays
ax = axes[0]
ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto', interpolation='nearest')
ax.axvline(gt['kerf_left_px'],  color='lime',   lw=1.5, ls='--', label=f'GT left  {gt["kerf_left_px"]:.1f}px')
ax.axvline(gt['kerf_right_px'], color='lime',   lw=1.5, ls='--', label=f'GT right {gt["kerf_right_px"]:.1f}px')
if res_w['status'] == 'ok':
    ax.axvline(res_w['left_edge_px'],  color='red',  lw=1.5, label=f'Det left  {res_w["left_edge_px"]}px')
    ax.axvline(res_w['right_edge_px'], color='cyan', lw=1.5, label=f'Det right {res_w["right_edge_px"]}px')
if res_t['status'] == 'ok':
    ax.scatter([res_t['tip_x']], [res_t['tip_y']], c='yellow', s=80, zorder=5, label=f'Tip y={res_t["tip_y"]:.0f}')
ax.set_title('Synthetic kerf (128×256) + detections')
ax.legend(fontsize=7, loc='lower right')
ax.set_xlabel('column [px]'); ax.set_ylabel('row [px]')

# Right: column projection
from vision._kerf_detection_kernel import detect_kerf_width as _cpp_detect  # for proj debug
import importlib
try:
    from scipy.ndimage import sobel
    gx = np.abs(sobel(img, axis=1))
    proj = gx.sum(axis=0)
    axes[1].plot(proj, lw=1, color='steelblue')
    axes[1].axhline(5000.0, color='orange', ls=':', lw=1, label='threshold=5000')
    axes[1].axvline(gt['kerf_left_px'],  color='lime', ls='--', lw=1.5)
    axes[1].axvline(gt['kerf_right_px'], color='lime', ls='--', lw=1.5, label='GT walls')
    axes[1].set_title('Sobel-X column projection')
    axes[1].set_xlabel('column [px]'); axes[1].set_ylabel('projection value')
    axes[1].legend(fontsize=8)
except ImportError:
    axes[1].text(0.5, 0.5, 'scipy not available', ha='center', transform=axes[1].transAxes)

plt.suptitle(
    f"GT: {gt['kerf_width_mm']:.2f} mm  |  "
    f"Det: {res_w.get('kerf_width_mm', -1):.2f} mm  |  "
    f"Chipping: {res_c['chipping_ratio']*100:.1f}%",
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'notebooks', 'vision_single_kerf.png'), dpi=130, bbox_inches='tight')
plt.show()

## 2 — Parameter sweep: kerf width × chipping × blade wear

In [ ]:
variants = [
    ('Clean (nominal)',     dict(kerf_width_mm=2.0, chipping_severity=0.0, blade_wear=0.0, include_blade_tip=False)),
    ('With blade tip',      dict(kerf_width_mm=2.0, chipping_severity=0.0, blade_wear=0.0, include_blade_tip=True)),
    ('Light chipping',      dict(kerf_width_mm=2.0, chipping_severity=0.3, blade_wear=0.0, include_blade_tip=False)),
    ('Heavy chipping',      dict(kerf_width_mm=2.0, chipping_severity=0.8, blade_wear=0.0, include_blade_tip=False)),
    ('Worn blade (σ×3)',    dict(kerf_width_mm=2.0, chipping_severity=0.0, blade_wear=1.0, include_blade_tip=False)),
    ('Narrow kerf (0.8mm)', dict(kerf_width_mm=0.8, chipping_severity=0.1, blade_wear=0.0, include_blade_tip=False)),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (title, kwargs) in zip(axes.flat, variants):
    cfg = KerfImageConfig(H=128, W=256, pixel_per_mm=10.0, seed=0, **kwargs)
    img, gt = generate_kerf_image(cfg)
    res = detect_kerf_width(img, threshold=5000.0, pixel_per_mm=10.0)
    ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto', interpolation='nearest')
    ax.axvline(gt['kerf_left_px'],  color='lime', lw=1.2, ls='--')
    ax.axvline(gt['kerf_right_px'], color='lime', lw=1.2, ls='--')
    if res['status'] == 'ok':
        ax.axvline(res['left_edge_px'],  color='red',  lw=1)
        ax.axvline(res['right_edge_px'], color='cyan', lw=1)
        det_str = f"Det={res['kerf_width_mm']:.2f}mm"
    else:
        det_str = 'Det=N/A'
    ax.set_title(f"{title}\nGT={gt['kerf_width_mm']:.2f}mm  {det_str}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Synthetic kerf parameter sweep (green=GT, red/cyan=detected)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'notebooks', 'vision_param_sweep.png'), dpi=130, bbox_inches='tight')
plt.show()

## 3 — Detection accuracy on 100-image synthetic dataset

In [ ]:
dataset = make_dataset(n_images=100, seed=7)

gt_widths, det_widths, errors = [], [], []
n_failed = 0
for sample in dataset:
    img = sample['image']
    gt  = sample['ground_truth']
    res = detect_kerf_width(img, threshold=5000.0, pixel_per_mm=10.0)
    if res['status'] == 'ok':
        gt_widths.append(gt['kerf_width_mm'])
        det_widths.append(res['kerf_width_mm'])
        errors.append(abs(res['kerf_width_mm'] - gt['kerf_width_mm']))
    else:
        n_failed += 1

errors = np.array(errors)
print(f'Detected: {len(det_widths)}/100 | Failed: {n_failed}')
print(f'MAE:  {errors.mean():.3f} mm')
print(f'P90 error: {np.percentile(errors, 90):.3f} mm')
print(f'Max error: {errors.max():.3f} mm')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: detected vs ground truth
ax = axes[0]
sc = ax.scatter(gt_widths, det_widths, c=errors, cmap='RdYlGn_r', vmin=0, vmax=1.0, s=20, alpha=0.8)
lims = [0, 4.5]
ax.plot(lims, lims, 'k--', lw=1, label='ideal')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('GT kerf width [mm]'); ax.set_ylabel('Detected kerf width [mm]')
ax.set_title(f'Detection accuracy (n={len(det_widths)})')
ax.legend()
plt.colorbar(sc, ax=ax, label='|error| [mm]')

# Error histogram
ax = axes[1]
ax.hist(errors, bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(errors.mean(), color='red', lw=2, label=f'MAE={errors.mean():.3f} mm')
ax.axvline(np.percentile(errors, 90), color='orange', lw=1.5, ls='--',
           label=f'P90={np.percentile(errors,90):.3f} mm')
ax.set_xlabel('|error| [mm]'); ax.set_ylabel('count')
ax.set_title('Detection error distribution')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(REPO_ROOT, 'notebooks', 'vision_accuracy.png'), dpi=130, bbox_inches='tight')
plt.show()

## 4 — NEU Steel dataset (scratches → kerf edge analog)

**To enable this cell**, download the NEU Surface Defect Database and set `NEU_ROOT`:

```bash
# Option A: Kaggle
kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database
unzip neu-surface-defect-database.zip -d ~/data/NEU-CLS

# Option B: Original site
# http://faculty.neu.edu.cn/yunhyan/NEU_surface_defect_database.html
```

Then set `NEU_ROOT = "/path/to/NEU-CLS"` below.

In [ ]:
NEU_ROOT = os.path.expanduser('~/data/NEU-CLS')  # ← change this path

try:
    adapter = NeuSteelAdapter(NEU_ROOT)
    print(f'NEU categories found: {adapter.available_categories}')
    _HAS_NEU = True
except FileNotFoundError as e:
    print(f'[SKIP] {e}')
    _HAS_NEU = False

if _HAS_NEU:
    # Load scratches (best kerf analog) + crazing (chipping analog)
    scratch_imgs  = adapter.as_kerf_input('scratches',     max_images=6)
    crazing_imgs  = adapter.as_kerf_input('crazing',       max_images=6)
    pitting_imgs  = adapter.as_kerf_input('pitted_surface', max_images=6)

    fig, axes = plt.subplots(3, 6, figsize=(14, 7))
    categories = [
        ('scratches → kerf edge',    scratch_imgs,  'lime'),
        ('crazing → dist. chipping', crazing_imgs,  'orange'),
        ('pitting → point chip',     pitting_imgs,  'cyan'),
    ]
    for row_idx, (label, imgs, color) in enumerate(categories):
        for col_idx, img in enumerate(imgs[:6]):
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto', interpolation='nearest')
            res = detect_kerf_width(img, threshold=2000.0, pixel_per_mm=10.0)
            if res['status'] == 'ok':
                ax.axvline(res['left_edge_px'],  color=color, lw=1)
                ax.axvline(res['right_edge_px'], color='white', lw=1, ls='--')
            ax.set_xticks([]); ax.set_yticks([])
        axes[row_idx, 0].set_ylabel(label, fontsize=8)

    plt.suptitle('NEU Steel Dataset → kerf inspection (detect_kerf_width overlaid)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(REPO_ROOT, 'notebooks', 'vision_neu_steel.png'), dpi=130, bbox_inches='tight')
    plt.show()
else:
    print('Showing synthetic substitutes for NEU dataset (download NEU-CLS to use real images):')
    mappings = [
        ('scratches → kerf edge',    dict(chipping_severity=0.0, blade_wear=0.0)),
        ('crazing → dist. chipping', dict(chipping_severity=0.7, blade_wear=0.0)),
        ('pitting → point chip',     dict(chipping_severity=0.5, blade_wear=0.3)),
    ]
    fig, axes = plt.subplots(3, 6, figsize=(14, 7))
    for row_idx, (label, kwargs) in enumerate(mappings):
        for col_idx in range(6):
            cfg = KerfImageConfig(H=128, W=256, pixel_per_mm=10.0, seed=col_idx * 10 + row_idx, **kwargs)
            img, gt = generate_kerf_image(cfg)
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto', interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
        axes[row_idx, 0].set_ylabel(label + '\n(synthetic)', fontsize=8)
    plt.suptitle('Synthetic substitutes (download NEU-CLS for real dataset)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(REPO_ROOT, 'notebooks', 'vision_neu_steel_synthetic.png'), dpi=130, bbox_inches='tight')
    plt.show()

## 5 — Summary

| Component | Module | Tested on |
|---|---|---|
| `detect_kerf_width` | `vision/_kerf_detection_kernel.so` (C++14) | 100 synthetic images |
| `detect_blade_tip` | same kernel | single image |
| `measure_chipping` | same kernel | per-sample |
| Synthetic data gen | `vision/generate_synthetic_kerf.py` | physical model |
| NEU Steel adapter | `vision/neu_steel_adapter.py` | scratch/crazing/pitting |

**Why no public kerf dataset exists:** DISCO/ADT blade-camera feeds contain process IP (blade wear rates, dicing recipes, wafer quality). The synthetic generator reproduces the key optical features (specular wall reflection, kerf void, chipping shadow) from first principles.

**NEU Steel transfer rationale:** Surface defect features in steel (scratches = linear bright/dark edges, crazing = distributed crack patterns) share gradient statistics with kerf edges on Si. The `detect_kerf_width` Sobel-X projection works on NEU scratches with threshold ~2000 (lower than Si because steel has higher surface contrast).